# Module 00: LLM Foundations

**Goal:** Build the mental models every LLM engineer needs before writing production code.

**What you'll learn:**
- What tokens are and why they're the fundamental unit of LLM cost
- Embeddings as geometry — why semantic search works
- Context window anatomy and its hard limits
- Sampling parameters: temperature, top-p, max_tokens
- Responses API (recommended) vs Chat Completions
- Model selection and cost estimation

**Prerequisites:** Set up your `.env` file with an API key. See `SETUP.md` at the repo root.

---

## 0. Setup

In [ ]:
%pip install -q openai anthropic tiktoken python-dotenv numpy

In [ ]:
import os
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(dotenv_path="../.env")

client = OpenAI()

def embed(text: str) -> list[float]:
    response = client.embeddings.create(model="text-embedding-3-small", input=text)
    return response.data[0].embedding

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print("✅ Setup complete")

---
## 1. Tokens — The Atomic Unit of LLMs

Tokens are chunks that a tokenizer has learned tend to appear together. They're the unit of cost, context, and capacity.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")  # GPT-4, Claude, most modern models

examples = [
    "Hello",
    "Hello world",
    "The quick brown fox jumps over the lazy dog.",
    "antidisestablishmentarianism",
    "🎉🎊🥳",
    "def fibonacci(n): return n if n <= 1 else fibonacci(n-1)+fibonacci(n-2)",
    "你好，世界",
]

print(f"  {'Text':<55} {'Tokens':>6}  {'Tok/char':>8}")
print("  " + "─" * 72)
for text in examples:
    tokens = enc.encode(text)
    ratio = len(tokens) / len(text)
    display = text if len(text) < 55 else text[:52] + "..."
    print(f"  {display:<55} {len(tokens):>6}   {ratio:>7.2f}")

**Key insight:** The model cannot count characters. Ask it "how many letters in 'strawberry'?" and it will often fail — because it never sees individual characters, only token chunks.

**Pricing is per token.** A 10,000-word document is ~13,000 tokens.

---
## 2. Embeddings — The Geometry of Meaning

An embedding is a high-dimensional vector representing text meaning. Similar meanings → similar vectors.

In [ ]:
# Semantically related pairs (expect high similarity)
pairs_related = [
    ("The cat sat on the mat", "A feline rested on the rug"),
    ("How to fix a memory leak in Python", "Python garbage collection and OOM errors"),
]

# Unrelated pairs (expect low similarity)
pairs_unrelated = [
    ("The cat sat on the mat", "Q3 revenue grew 24% YoY"),
    ("How to fix a memory leak", "The history of the Roman Empire"),
]

print("Related pairs (expect high similarity):")
for a, b in pairs_related:
    score = cosine_similarity(embed(a), embed(b))
    print(f"  {score:.3f} | '{a[:35]}' ↔ '{b[:35]}'")

print("\nUnrelated pairs (expect low similarity):")
for a, b in pairs_unrelated:
    score = cosine_similarity(embed(a), embed(b))
    print(f"  {score:.3f} | '{a[:35]}' ↔ '{b[:35]}'")

---
## 3. API Call Anatomy

Every LLM call has: system prompt (instructions), conversation history, and the current user message.

In [ ]:
import time

start = time.time()
response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0.3,
    max_tokens=200,
    messages=[
        {"role": "system", "content": "You are a concise technical tutor. Explain in 2-3 sentences."},
        {"role": "user", "content": "What is a transformer?"}
    ]
)
latency_ms = (time.time() - start) * 1000

print(f"Response: {response.choices[0].message.content[:150]}...")
print(f"Tokens in: {response.usage.prompt_tokens}")
print(f"Tokens out: {response.usage.completion_tokens}")
print(f"Latency: {latency_ms:.0f}ms")

---
## 4. Temperature Effect

Temperature controls randomness. 0.0 = deterministic, 1.0 = creative/varied.

In [ ]:
prompt = "Give me one creative name for a coffee shop."

print("temperature=0.0 (deterministic):")
for _ in range(3):
    r = client.chat.completions.create(
        model="gpt-4o-mini", temperature=0.0, max_tokens=30,
        messages=[{"role": "user", "content": prompt}]
    )
    print(f"  '{r.choices[0].message.content.strip()}'")

print("\ntemperature=1.0 (varied):")
for _ in range(3):
    r = client.chat.completions.create(
        model="gpt-4o-mini", temperature=1.0, max_tokens=30,
        messages=[{"role": "user", "content": prompt}]
    )
    print(f"  '{r.choices[0].message.content.strip()}'")

---
## 5. Cost Estimation

Always estimate cost before building. Input is cheap; output is expensive.

In [ ]:
pricing = {
    "gpt-4o-mini":     (0.15,  0.60),
    "gpt-4o":          (2.50, 10.00),
    "claude-haiku-4-5":(0.80,  4.00),
    "claude-sonnet-5": (3.00, 15.00),
    "claude-opus-4-8": (5.00, 25.00),
}

print(f"  {'Model':<25} | {'Input/1M':>10} | {'Output/1M':>10}")
print("  " + "─" * 50)
for model, (inp, out) in pricing.items():
    print(f"  {model:<25} | ${inp:>9.2f} | ${out:>9.2f}")

print("\nScale: 5,000 conversations/day, 500-word context, 150-word response:")
for model in ["gpt-4o-mini", "gpt-4o"]:
    inp, out = pricing[model]
    cost_per_call = (650 / 1e6 * inp) + (195 / 1e6 * out)
    daily = cost_per_call * 5000
    print(f"  {model:<25} | ${daily:.2f}/day | ${daily * 30:.0f}/month")

---
## Summary

| Concept | Key Takeaway |
|---------|-------------|
| **Tokens** | Unit of cost, context, and capacity |
| **Embeddings** | Meaning as geometry; powers semantic search |
| **Context Window** | Stateless; re-send everything each call |
| **Temperature** | 0.0 for deterministic, 0.7 for creative |
| **Cost** | Input is cheap, output is expensive; right-size your model |

**Next:** Module 01 — Prompt Engineering, where you'll use these fundamentals to craft effective LLM inputs.

---
## 🧪 Exercises

1. **Token Counting**: Use tiktoken to count tokens in 10 different inputs — a tweet, a paragraph, a JSON object, a Python function, an emoji-heavy string. Which has the highest tokens-per-character ratio?

2. **Embedding Space**: Embed 5 sentences on the same topic and 5 on different topics. Compute pairwise similarities. Do within-topic pairs always score higher?

3. **Temperature Experiment**: Run the same creative prompt 3 times at temperature 0.0, then 3 times at temperature 0.8. How many unique responses at each?

4. **Cost Calculator**: Build a function `estimate_cost(model, input_words, output_words)` that returns dollar cost. Estimate daily cost for 5,000 conversations/day.